# HW2 — Ocean heat uptake and regional heat transport

Overview of the homework:

1. Build a model with separate surface and deep-ocean temperatures, and compare it with simpler representations of the ocean.
2. Diagnose regional heat transport in the latitude-dependent model.

Use NumPy, Matplotlib, and SciPy (as in the labs or HWs). Cells marked **TODO** need your code; written responses go in the answer cells. Most plotting code is supplied. As in HW1, using AI to help with plotting is fair game, but avoid using AI to solve the computing/modeling aspects (except to debug code).

Guess what will happen before running experiments. Keep your original guess and explain any change in your understanding. A few sentences per written question are sufficient unless otherwise stated. Show the main steps in algebra and include units. Guesses are assessed for reasoning, not whether they turn out to be correct. Partial credit is available for intermediate steps.

**Grading: 100 points.**

| Part | Points |
|:--|--:|
| 1. Connect surface and deep-ocean reservoirs (1a–1d) | 55 |
| 2. Regional heat transport (2a–2d) | 45 |

All equations and parameters needed are included below. No downloads are required. 

**Submission:** save the completed notebook with outputs in your private course repository and submit its link through Courseworks. See the assignment on Courseworks for submission dates.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({'figure.figsize': (8, 4.5), 'font.size': 11})

## 1. Connect surface and deep-ocean reservoirs

In HW1, a single heat capacity connected temperature change to stored energy. Here we separate an effective surface reservoir (upper ocean and rapidly coupled atmosphere and land) from the deep ocean:

$$C_s\frac{dT_s'}{dt}=F+\alpha T_s'-Q',\qquad
C_d\frac{dT_d'}{dt}=Q',\qquad
Q'=\gamma(T_s'-T_d').$$

The derivatives $dT_s'/dt$ and $dT_d'/dt$ are the **rates of temperature change**, sometimes called **temperature tendencies**. A positive rate means warming; a negative rate means cooling. Their units are K/s when time is in seconds. For example, a rate of +0.1 K/year means the temperature is increasing by 0.1 K per year at that instant.

Both temperatures are **changes from each reservoir's own reference temperature**. Positive $Q'$ means additional heat transfer from surface to deep ocean. Equal anomalies mean equal warming, not equal absolute temperatures. The deep box exchanges energy with the surface, with no direct exchange with space.

We use the forcing–feedback relationship from class to calculate net incoming radiation: $N'=F+\alpha T_s'$. Positive $N'$ means net energy enters the whole system. 

HW1 used $\lambda>0$ for restoring strength; here $\alpha=-\lambda<0$ is the signed feedback parameter used in the recent slides and following same sign convention as the IPCC report. We hold it fixed throughout this homework.

In this HW we primarily use these parameter values:
| Parameter | Value | Meaning |
|:--|--:|:--|
| $F$ (constant) | 4 W m⁻² | Imposed additional energy input |
| $\alpha$ | −1 W m⁻² K⁻¹ | Fixed radiative feedback |
| $C_s$ | $2.2\times10^8$ J m⁻² K⁻¹ | Surface heat capacity |
| $C_d$ | $3.2\times10^9$ J m⁻² K⁻¹ | Deep-ocean heat capacity |
| $\gamma$ | 0.7 W m⁻² K⁻¹ | Exchange coefficient |

All fluxes and heat capacities are expressed **per square metre of Earth’s total surface area**. For example, $C_d$ is the deep ocean’s total heat capacity divided by Earth’s total surface area, rather than by the ocean area alone. The supplied values already use this convention; you do not need to apply an ocean-area fraction. Time in the equations is in seconds. Use 365 days per year, as in HW1.

### 1a. Two connected budgets (10 points)

Start with both temperature anomalies ($T'_d$ and $T'_s$) equal to zero and maintain $F=4$ W m⁻².

1. For $N'>0$, which box receives the net energy input from outside the system? For $Q'>0$, which box loses energy through exchange and which gains it? Then add the two budgets to obtain an equation for total stored energy $E'=C_sT_s'+C_dT_d'$. Explain why the exchange terms cancel.
2. Calculate each box's warming rate immediately after the forcing begins, in K/year. Explain why deep warming develops after the surface starts warming, even though its initial rate of temperature change is zero.
3. Set both rates of temperature change to zero - assuming that the system has reached equilibrium. For $\gamma>0$, what must be true of the two warming anomalies at equilibrium? Calculate their common value.

**Your response:**

_Write here._

In [ ]:
SECONDS_PER_YEAR = 365 * 24 * 60 * 60
F_on = 4.0
alpha = -1.0
Cs = 2.2e8
Cd = 3.2e9
gamma = 0.7

### 1b. Calculate both rates of temperature change (15 points)

Complete the function `tendencies`: calculate the exchange and each box’s rate of temperature change **in K/s**. It should accept either numbers or NumPy arrays.

The function inputs correspond to the equation symbols as follows:

| Code name | Equation symbol | Meaning | Units |
|:--|:--|:--|:--|
| `Ts` | $T_s'$ | Current surface temperature anomaly | K |
| `Td` | $T_d'$ | Current deep-ocean temperature anomaly | K |
| `forcing_now` | $F$ | Constant imposed forcing | W m⁻² |
| `exchange` | $\gamma$ | Exchange coefficient, not the heat-transfer flux itself | W m⁻² K⁻¹ |

The constants `Cs`, `Cd`, and `alpha` were defined in the preceding code cell. They correspond to $C_s$, $C_d$, and $\alpha$; use them directly inside the function.

Inside your function, use `Q` for the exchange flux $Q'$ in W m⁻². Return `dTs_dt` and `dTd_dt`, corresponding to $dT_s'/dt$ and $dT_d'/dt$, in that order. The code names `Ts` and `Td` omit the primes, but still represent anomalies, not absolute temperatures.

The supplied loop extends HW1's forward Euler method to two temperatures. Both updates use the **same old state**. Conversion to seconds is already in the loop; do not repeat it in `tendencies`. The `forcing` argument to `run_model` is a constant energy flux in W m⁻², set to 4 W m⁻² by default.

In [ ]:
def tendencies(Ts, Td, forcing_now, exchange):
    # Inputs: Ts = T_s′, Td = T_d′, forcing_now = F, exchange = gamma.
    # Use the constants Cs, Cd, and alpha defined above.
    # Return surface rate first, then deep-ocean rate, both in K/s.
    # TODO: calculate Q, dTs_dt, and dTd_dt from the two budgets.
    raise NotImplementedError("Complete the two-box tendencies.")
    return dTs_dt, dTd_dt


In [ ]:
def run_model(forcing=F_on, exchange=gamma, years=2000,
              dt_years=0.05, initial=(0.0, 0.0)):
    nsteps = round(years / dt_years)
    time = np.linspace(0, years, nsteps + 1)
    dt_seconds = (time[1] - time[0]) * SECONDS_PER_YEAR
    Ts = np.zeros(nsteps + 1)
    Td = np.zeros(nsteps + 1)
    Ts[0], Td[0] = initial
    for n in range(nsteps):
        dTs_dt, dTd_dt = tendencies(Ts[n], Td[n], forcing, exchange)
        Ts[n + 1] = Ts[n] + dt_seconds * dTs_dt
        Td[n + 1] = Td[n] + dt_seconds * dTd_dt
    return {'time': time, 'Ts': Ts, 'Td': Td, 'exchange': exchange,
            'forcing': forcing}

def at_year(run, year):
    return np.argmin(abs(run['time'] - year))

two_box = run_model()
finer = run_model(dt_years=0.025)
print(f"Maximum surface difference on halving the time step: "
      f"{np.max(abs(two_box['Ts'] - finer['Ts'][::2])):.4f} K")


### 1c. How does the deep ocean change surface warming? (15 points)

Use the numerical solver from 1b to compare two runs:

- **Disconnected deep ocean:** `exchange=0`. The surface evolves on its own, and the deep-ocean anomaly stays at zero.
- **Connected deep ocean:** `exchange=gamma`, using the original exchange coefficient of 0.7 W m⁻² K⁻¹.

Both runs start with zero temperature anomalies and use the same constant forcing, feedback parameter, and heat capacities. Only the exchange coefficient changes. The calculation and plotting code below is supplied.

1. **Before running:** predict which surface will be warmer after 50 years. Explain your reasoning using the transfer of energy between the boxes.
2. Run the code and report both surface temperature anomalies at year 50. Do the results support your prediction? Explain why the two warming histories differ.
3. Does connecting the deep ocean change the eventual surface temperature anomaly? Use your equilibrium result from 1a and the long-time plot to explain. Distinguish the eventual amount of surface warming from how quickly it is reached.

**Your response:**

_Write here._

In [ ]:
disconnected = run_model(exchange=0.0)
connected = two_box  # Reuse the original connected run from 1b.

fig, axes = plt.subplots(1, 2, figsize=(11, 4), layout='constrained')
for ax, horizon in zip(axes, [50, 2000]):
    for run, label in [(disconnected, 'Disconnected deep ocean'),
                       (connected, 'Connected deep ocean')]:
        keep = run['time'] <= horizon
        ax.plot(run['time'][keep], run['Ts'][keep], label=label)
    ax.set(xlabel='Year after forcing begins', ylabel='Surface warming (K)')
    ax.legend(fontsize=8)
plt.show()
for run, label in [(disconnected, 'Disconnected'), (connected, 'Connected')]:
    i = at_year(run, 50)
    print(f"{label}, year 50: surface warming = {run['Ts'][i]:.3f} K")


### 1d. Change ocean exchange (15 points)

Hold all parameters fixed except $\gamma$. Compare weak exchange, $\gamma=0.35$, with stronger exchange, $\gamma=1.4$ W m⁻² K⁻¹.

**Before running:** predict which surface will be warmer at year 50 and which deep box will have accumulated more energy by then.

Use the printed values and plots to check your predictions. Explain both comparisons using $Q'=\gamma(T_s'-T_d')$. Does increasing the exchange coefficient by a factor of four necessarily increase the exchange flux by a factor of four in a later state? Explain why the temperature-anomaly difference matters. Your argument should distinguish a comparison at year 50 from the common eventual equilibrium.

**Your response:**

_Write here._

In [ ]:
weak = run_model(exchange=0.35)
strong = run_model(exchange=1.4)
fig, axes = plt.subplots(1, 2, figsize=(11, 4), layout='constrained')
for run, label in [(weak, 'Weak exchange'), (strong, 'Stronger exchange')]:
    keep = run['time'] <= 150
    axes[0].plot(run['time'][keep], run['Ts'][keep], label=label)
    axes[1].plot(run['time'][keep], Cd * run['Td'][keep] / 1e9, label=label)
    i = at_year(run, 50)
    Q = run['exchange'] * (run['Ts'][i] - run['Td'][i])
    print(f"{label}, year 50: Ts={run['Ts'][i]:.3f} K; Td={run['Td'][i]:.3f} K; "
          f"deep energy={Cd*run['Td'][i]/1e9:.3f} GJ m⁻²; Q={Q:.3f} W m⁻²")
for ax in axes:
    ax.set_xlabel('Year after forcing begins')
    ax.legend(fontsize=9)
axes[0].set_ylabel('Surface warming (K)')
axes[1].set_ylabel('Deep stored-energy change (GJ m$^{-2}$)')
plt.show()


## 2. Regional heat transport

We now use the **latitude-dependent model from the October 15 lab**. It divides Earth into 90 bands, each 2° wide, and follows the temperature in each band. Heat can move between neighboring bands. We keep the lab's constant albedo and its earlier expression for outgoing infrared radiation:

$$C\frac{dT_i}{dt}=N_i+h_i,\qquad N_i=\mathrm{ASR}_i-\mathrm{OLR}_i,$$
$$\mathrm{ASR}_i=(1-0.30)Q_i,\qquad \mathrm{OLR}_i=(1-\epsilon/2)\sigma T_i^4.$$

Here $i$ labels a latitude band. ASR means **absorbed sunlight**, OLR means **outgoing infrared radiation**, and $N_i$ is their difference. A positive $N_i$ supplies energy through radiation. $Q_i$ is annual-mean incoming sunlight in that band, already averaged over day and year; do not divide it by four again. Here $Q_i$ denotes sunlight, whereas $Q'$ in Part 1 denoted exchange with the deep ocean.

$T_i$ is **absolute temperature in kelvin** in the radiation calculation; plots show °C. A change of 1 K has the same size as a change of 1°C. The rate $dT_i/dt$ is positive for warming and negative for cooling.

| Quantity | Meaning | Value or units |
|:--|:--|:--|
| $\epsilon$ | Atmospheric infrared absorptivity/emissivity, as in HW1 | 0.779 |
| $\sigma$ | Stefan–Boltzmann constant | $5.670374419\times10^{-8}$ W m⁻² K⁻⁴ |
| $C$ | Heat capacity per square metre of each band, equivalent to a 10 m water layer | $4.18\times10^7$ J m⁻² K⁻¹ |
| $D$ | Transport coefficient: larger values strengthen exchange for a given temperature gradient | W m⁻² K⁻¹ |

In this section, fluxes such as $N_i$ and $h_i$ and the capacity $C$ are per square metre **of the local band**. To calculate energy transfer for the whole band, multiply a local flux by that band's area.

Let $H$ be the rate of heat transfer across an entire latitude circle, in W. Positive $H$ means northward transfer; negative $H$ means southward transfer. **1 PW = $10^{15}$ W.** The heating from transport in a band is

$$h_i=\frac{H_{i,\mathrm{south}}-H_{i,\mathrm{north}}}{a_i},$$

where $a_i$ is the band area in m². This difference between incoming and outgoing transfer is called **transport convergence**. Positive $h_i$ heats the band; negative $h_i$ cools it. Both $h_i$ and $N_i$ are in W m⁻². Their sum $N_i+h_i=C\,dT_i/dt$ is the **rate of heat storage per unit area**. Steady temperature requires this sum to be zero, even when its individual terms are nonzero.

Heat cannot cross either pole, so $H=0$ at both polar boundaries. Internal transfers leave one band and enter its neighbor. The bands have different areas, so a global mean is weighted by their areas: multiply each value by its band's fraction of Earth's area, then add. The supplied `global_mean` function does this calculation.

All solar and transport calculations and the time integration are supplied. No external data are needed. Use 365 days per year throughout this homework, as in HW1. The supplied code uses the same `SECONDS_PER_YEAR` constant in both parts.

### 2a. Direction of transport and local heating (10 points)

For a band in the Northern Hemisphere with area $5\times10^{13}$ m², northward heat transfer is +1.0 PW at the southern boundary and +1.5 PW at the northern boundary. In this example, heat enters through the southern boundary and leaves through the northern boundary. You may calculate the answers by hand or in Python; show the main steps and units.

1. Calculate $h$. Is the band heated or cooled by transport? Explain how this can happen when heat is moving northward at both boundaries.
2. If the band's temperature is steady, what must its net radiation $N$ be? Explain how a steady region can have nonzero net radiation.
3. Repeat the calculation of $h$ when both boundary transfers are +1.5 PW. Does a large transfer through a region necessarily heat that region? Explain.

**Your response:**

_Write here._

### 2b. Calculate transport from the radiation budget (15 points)

**Why do this?** Scientists use satellite measurements of Earth’s radiation budget to estimate how much energy the atmosphere and ocean together transport between latitudes. This exercise uses the same energy-budget logic: work out the heat transport needed to balance regional gains and losses of radiation, without calculating every wind and ocean current. Here we use our model’s radiation budget and assume temperatures are steady; real-world estimates also account for changing heat storage and measurement uncertainty. See [Fasullo and Trenberth (2008)](https://doi.org/10.1175/2007JCLI1936.1) for an application using satellite radiation measurements.

Run the supplied setup and calculation cells below. They use $D=0.6$ W m⁻² K⁻¹ and start every band at 288 K. The model transfers heat toward colder neighboring bands, as in the lab.

The first calculation prints the largest final $|N+h|$ among all bands. This is the largest remaining rate of heat storage per unit area. For this exercise, a value below 0.01 W m⁻² is small enough to treat the final temperatures as steady.

In [ ]:
# Supplied model helpers from the latest lab: run this cell as written.
# No edits to the solar calculation or numerical solver are required.
from scipy.integrate import solve_ivp

R_EARTH = 6.371e6
RHO_CP = 1000.0 * 4180.0  # water density times specific heat: J/m³/K

# Ninety 2-degree latitude bands, from the South Pole to the North Pole.
phi_edges = np.linspace(-np.pi/2, np.pi/2, 91)
phi = 0.5 * (phi_edges[:-1] + phi_edges[1:])
lat, lat_edges = np.rad2deg(phi), np.rad2deg(phi_edges)
band_area = 2*np.pi*R_EARTH**2 * np.diff(np.sin(phi_edges))
weights = band_area / band_area.sum()

def global_mean(values):
    '''Area-weighted mean along the last (latitude) dimension.'''
    return np.asarray(values) @ weights

def annual_sunlight(phi, solar_constant=1361.0, tilt_deg=23.44):
    '''Daily solar geometry averaged over a circular orbit (720 phases).'''
    longitude = 2*np.pi*(np.arange(720) + 0.5)/720
    delta = np.arcsin(np.sin(np.deg2rad(tilt_deg))*np.sin(longitude))
    p = np.asarray(phi)[:, None]
    sunset = np.arccos(np.clip(-np.tan(p)*np.tan(delta), -1, 1))
    daily = solar_constant/np.pi * (
        sunset*np.sin(p)*np.sin(delta)
        + np.cos(p)*np.cos(delta)*np.sin(sunset))
    return np.maximum(daily, 0).mean(axis=1)

Q = annual_sunlight(phi)

def transport(T, D=0.6):
    '''Return northward boundary power H (W) and band heating h (W/m²).'''
    T = np.asarray(T, dtype=float)
    H = np.zeros(len(T) + 1)  # polar boundary powers remain zero
    gradient = np.diff(T) / np.diff(phi)
    H[1:-1] = -2*np.pi*R_EARTH**2 * np.cos(phi_edges[1:-1])*D*gradient
    h = (H[:-1] - H[1:]) / band_area
    return H, h


SIGMA = 5.670374419e-8
EPSILON = 0.779

def gray_olr(T_K, epsilon=EPSILON):
    return (1 - epsilon/2)*SIGMA*np.asarray(T_K)**4

def simulate_gray(D=0.0, albedo_value=0.30, epsilon=EPSILON,
                  years=30.0, water_depth=10.0, initial_K=288.0):
    C = RHO_CP*water_depth
    reflect = np.broadcast_to(np.asarray(albedo_value, dtype=float), lat.shape)
    if not (0 <= epsilon <= 1 and np.all((reflect >= 0) & (reflect <= 1))):
        raise ValueError('Emissivity and albedo must lie between 0 and 1.')
    if D < 0 or min(years, water_depth, initial_K) <= 0:
        raise ValueError('Use nonnegative D and positive time, depth and temperature.')
    absorbed = (1-reflect)*Q
    def rhs(t, T_K):
        h = np.zeros_like(T_K) if D == 0 else transport(T_K, D)[1]
        return SECONDS_PER_YEAR/C*(absorbed-gray_olr(T_K, epsilon)+h)
    time = np.linspace(0, years, 301)
    solution = solve_ivp(rhs, (0, years), np.full(len(lat), initial_K),
                         t_eval=time, method='BDF', rtol=1e-9, atol=1e-10)
    if not solution.success:
        raise RuntimeError(solution.message)
    return dict(time=time, T_K=solution.y.T, T=solution.y.T-273.15,
                D=D, C=C, ASR=absorbed, epsilon=epsilon)



In [ ]:
# Run the supplied latitude model for 30 years with transport coefficient D = 0.6.
regional = simulate_gray(D=0.6)
# The returned dictionary stores temperatures at all saved times.
# 'T_K' is temperature in kelvin; [-1] selects the final time.
T_regional = regional['T_K'][-1]
# H_gradient: boundary transfers (W); h_regional: band heating (W/m²).
H_gradient, h_regional = transport(T_regional, D=0.6)
# N = absorbed sunlight minus outgoing infrared radiation, in W/m².
N_regional = regional['ASR'] - gray_olr(T_regional)
# N + h is the rate of heat storage per unit area; near zero means nearly steady temperature.
print(f"Maximum final |N+h| = {np.max(abs(N_regional+h_regional)):.2e} W m⁻²")


**2b — complete the transport update**

At steady temperature, rearranging the band budget gives

$$H_{i,\mathrm{north}}=H_{i,\mathrm{south}}+a_iN_i.$$

Starting with zero transfer at the South Pole, add each band's net radiative input to calculate transfer at its northern boundary. Work through the bands **from south to north**. The northern boundary of one band is the southern boundary of the next.

Complete the update inside the supplied `infer_transport` loop:

| Code name | Equation quantity | Meaning and units |
|:--|:--|:--|
| `net_radiation[i]` | $N_i$ | Band $i$'s net radiative input, W m⁻² |
| `areas[i]` | $a_i$ | Area of band $i$, m² |
| `H_inferred[i]` | $H_{i,\mathrm{south}}$ | Transfer at its southern boundary, W |
| `H_inferred[i + 1]` | $H_{i,\mathrm{north}}$ | Transfer at its northern boundary, W |

There are 90 band values and **91 boundary values**. Array index 0 is the South Pole boundary; index 90 is the North Pole boundary. The first value is supplied as zero. Calculate the last value from the budget too, rather than setting it to zero.

In [ ]:
def infer_transport(net_radiation, areas):
    # Both input arrays are ordered from the southernmost to northernmost band.
    # net_radiation: N_i, in W/m². areas: a_i, in m².
    H_inferred = np.zeros(len(net_radiation) + 1)  # H at boundaries, in W
    # H_inferred[0] = 0 is the South Pole boundary condition.
    for i in range(len(net_radiation)):
        # TODO: calculate H_inferred[i + 1] from H_inferred[i], areas[i],
        # and net_radiation[i], using the steady band budget above.
        raise NotImplementedError("Complete the boundary-to-boundary update.")
    return H_inferred


In [ ]:
# Apply your function to the model’s final radiation budget and band areas.
H_inferred = infer_transport(N_regional, band_area)
# Compare your result with transport calculated from the temperature differences.
# Divide watts by 10^15 to express the large transfers in petawatts (PW).
print(f"Maximum reconstruction difference: {np.max(abs(H_inferred-H_gradient))/1e15:.3e} PW")
# The last boundary is the North Pole, where the physical transport is zero.
print(f"Reconstructed North Pole power: {H_inferred[-1]/1e15:+.3e} PW")
fig, axes = plt.subplots(1, 2, figsize=(11, 4), layout='constrained')
# Left panel: radiation and transport heating should balance at steady temperature.
axes[0].plot(lat, N_regional, label='Net radiation N')
axes[0].plot(lat, h_regional, label='Transport heating h')
# Left panel: radiation and transport heating should balance at steady temperature.
axes[0].plot(lat, N_regional+h_regional, '--', label='Storage rate N+h')
axes[0].set(xlabel='Latitude (°)', ylabel='Energy flux (W m$^{-2}$)')
axes[0].legend(fontsize=8)
# Right panel: compare the two ways of calculating transport at band boundaries.
# Use lat_edges (91 boundaries), rather than lat (90 band centers).
axes[1].plot(lat_edges, H_gradient/1e15, lw=3, label='From temperature gradient')
axes[1].plot(lat_edges, H_inferred/1e15, '--', label='Reconstructed from radiation')
axes[1].axhline(0, color='0.5', lw=0.8)
axes[1].set(xlabel='Boundary latitude (°)', ylabel='Northward transport (PW)')
axes[1].legend(fontsize=8)
plt.show()


**2b — interpret your results**

Report the maximum difference between your result and the model's transport calculated from temperature differences, in PW. Also report your North Pole value. Explain why both numbers should be near zero when the final temperatures are steady. Would using $N_i$ alone in this reconstruction be valid while a band was still warming? Explain using its full energy budget.

**Your response:**

_Write here._

### 2c. Change transport strength (10 points)

Compare $D=0.3$ and $D=1.2$ W m⁻² K⁻¹. The supplied code changes only $D$. Albedo, sunlight, heat capacity, and the **formula** for outgoing infrared radiation stay the same. Outgoing radiation itself can change as temperatures change.

Here **transport rule** means the model's formula for heat transfer between neighboring bands:

$$H_{\mathrm{edge}}=-2\pi R^2\cos\phi_{\mathrm{edge}}\,D\,
\frac{T_{\mathrm{north}}-T_{\mathrm{south}}}{\phi_{\mathrm{north}}-\phi_{\mathrm{south}}}.$$

$H_{\mathrm{edge}}$ is northward heat transfer across their shared boundary, in W; $R$ is Earth's radius; and $\phi$ is latitude in radians. The temperature difference divided by the latitude difference is the **temperature gradient**. The minus sign makes heat flow toward colder bands. This is the formula already implemented in the supplied `transport(T, D)` function; you do not need to derive or code it.

For this question, focus on two factors: **$D$ and the temperature gradient**. At a given boundary, the geographical factor $2\pi R^2\cos\phi_{\mathrm{edge}}$ stays fixed between runs. Increasing $D$ strengthens transfer for an unchanged gradient, but the gradient can change as temperatures adjust.

**Before running:** imagine increasing $D$ while the temperatures are initially unchanged. More heat would then move from warmer toward colder bands. Would this tend to bring their temperatures closer together or farther apart? Use that reasoning to predict which run will have the smaller equator-to-pole temperature difference.

The final two printed quantities, mean OLR and OLR evaluated at mean temperature, are for 2d. You do not need to calculate them yourself.

**Your prediction for 2c (before running):**

_Write here._

In [ ]:
regional_runs = {D: simulate_gray(D=D) for D in [0.3, 1.2]}
fig, ax = plt.subplots(layout='constrained')
for D, run in regional_runs.items():
    T = run['T_K'][-1]
    H_D, h_D = transport(T, D)
    residual = run['ASR'] - gray_olr(T) + h_D
    # Select the band centers closest to 1°N and 89°N.
    equator_index = np.argmin(abs(lat - 1.0))
    polar_index = np.argmin(abs(lat - 89.0))
    contrast = T[equator_index] - T[polar_index]
    print(f"D={D}: contrast={contrast:.3f} K; max |H|={np.max(abs(H_D))/1e15:.3f} PW; "
          f"mean={global_mean(T)-273.15:.3f} °C; max |N+h|={np.max(abs(residual)):.2e} W m⁻²")
    print(f"    Mean OLR = {global_mean(gray_olr(T)):.3f} W m⁻²; "
          f"OLR at mean temperature = {gray_olr(global_mean(T)):.3f} W m⁻²")
    ax.plot(lat, T-273.15, label=f'D={D}')
ax.set(xlabel='Latitude (°)', ylabel='Final surface temperature (°C)')
ax.legend()
plt.show()


**2c — interpret your results step by step**

First check that both printed maximum $|N+h|$ values are below 0.01 W m⁻², so the final temperatures can be treated as steady. Then work through the following:

1. Record each run's temperature difference between the **1°N and 89°N band centers**, largest absolute transport $|H|$ in PW, and area-weighted mean temperature in the response table below.
2. Look at the temperature curves. With stronger transport, do the low latitudes become warmer or cooler? What happens at high latitudes? Does the temperature profile become flatter or steeper? This tells you how the **temperature gradient**, the change in temperature with latitude, responds.
3. Calculate the ratio of the two transport coefficients, $1.2/0.3$, and the ratio of the two printed maximum $|H|$ values (stronger divided by weaker). If the temperature profile stayed unchanged, the transport rule would make $|H|$ increase by the same factor as $D$. Does your calculated transport ratio match that factor?
4. Bring the pieces together: the transport rule depends on **both $D$ and the temperature gradient**. Use your answers to 2 and 3 to explain why changing $D$ alone does not determine the factor by which the transported heat changes. A few sentences are sufficient.

**Your response:**

| $D$ (W m⁻² K⁻¹) | Temperature difference (K) | Maximum transport magnitude (PW) | Global mean temperature (°C) |
|:--|--:|--:|--:|
| 0.3 | | | |
| 1.2 | | | |

**2. Temperature profile:** _Write here._

**3. Ratios:** _Write here._

**4. Explanation:** _Write here._

### 2d. Can redistribution change the global mean temperature? (10 points)

A classmate says: “Transport only moves heat internally, so changing transport strength must leave the final global mean temperature exactly unchanged.”

Use the results from 2c to respond in a few sentences:

1. Does the global mean temperature stay exactly the same in the two runs? Do their global mean outgoing radiation values stay approximately the same?
2. Absorbed sunlight is unchanged between these runs. When you add all band energy budgets, internal heat transfers cancel. At steady temperature, what does this require of total outgoing radiation? Does this explain the similar mean OLR values you found in part 1 of this question?
3. Compare the printed **mean OLR** with **OLR evaluated at mean temperature**. Are these the same calculation? Use the $T^4$ dependence to explain why the same total outgoing radiation can coexist with different global mean temperatures when the latitude profile changes.

A few sentences using the printed results and energy budget are sufficient. All fourth-power calculations in the supplied code use kelvin.

**Your response:**

_Write here._

## Before submitting

- Complete code, predictions, and explanations. Label plots and numerical results with units.
- Restart the kernel and run all cells in order.
- Save the notebook with outputs visible.